Elaprase Treated Patient Counts - 6522

In [0]:
create or replace temporary view MPSII_TREATMENT_TABLE as (SELECT * 
    FROM (SELECT DISTINCT PATIENT_ID AS PATIENT_ID,
                 COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
                 NDC11 AS CODE,
                 MEDICAL_EVENT_ID AS EVENT_ID,
                 SERVICE_DATE AS FILL_DATE,
                 PLACE_OF_SERVICE,
                 KH_PLAN_ID AS KH_PLAN,
                'MEDICAL_EVENTS' AS TABLE_NAME

         FROM com_edp_prd.com_raw.kom_medical_events
         WHERE NDC11 IN ('54092070001','540920700')
    UNION
         SELECT DISTINCT PATIENT_ID AS PATIENT_ID,
                 PRESCRIBER_NPI AS NPI,
                 NDC11 AS CODE,
                 PHARMACY_EVENT_ID as EVENT_ID,
                 FILL_DATE,
                 NULL AS PLACE_OF_SERVICE,
                 COALESCE(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) AS KH_PLAN,
                 'PHARMACY_EVENTS' AS TABLE_NAME
           FROM com_edp_prd.com_raw.kom_pharmacy_events
           WHERE NDC11 IN ('54092070001','540920700')
           AND TRANSACTION_RESULT = 'PAID'
    UNION

         SELECT DISTINCT PATIENT_ID AS PATIENT_ID,
                RENDERING_NPI AS NPI,
                PROCEDURE_CODE AS CODE,   
                MEDICAL_EVENT_ID AS EVENT_ID,
                SERVICE_DATE AS FILL_DATE, 
                PLACE_OF_SERVICE,
                KH_PLAN_ID AS KH_PLAN,
                'MEDICAL_EVENTS' AS TABLE_NAME


           FROM com_edp_prd.com_raw.kom_medical_events 
           WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379','38206','38230','38232','38240','38241','38242','38243','38250')
)
-- WHERE FILL_DATE BETWEEN '2020-04-01' AND '2025-03-31'
WHERE FILL_DATE BETWEEN '2020-08-01' AND '2025-07-31');

SELECT COUNT(DISTINCT PATIENT_ID) FROM MPSII_TREATMENT_TABLE

MPS II 1 Dx - 2928

In [0]:
create or replace temporary view MPSII_1Dx_Specified AS 
(
  SELECT * FROM (
    SELECT DISTINCT 
      PATIENT_ID,
      COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
      SERVICE_DATE AS FILL_DATE,
      MEDICAL_EVENT_ID AS EVENT_ID,
      DIAGNOSIS_CODES,
      KH_PLAN_ID AS KH_PLAN,
      Place_of_service
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E761%'

    UNION

    -- Pharmacy Events
    SELECT DISTINCT 
      PATIENT_ID,
      PRESCRIBER_NPI AS NPI,
      FILL_DATE,
      PHARMACY_EVENT_ID AS EVENT_ID,
      DIAGNOSIS_CODE AS DIAGNOSIS_CODES,
      COALESCE(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) AS KH_PLAN,
      NULL as Place_of_service
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E761'
      AND TRANSACTION_STATUS = 'PAID'
  ) AS combined
  WHERE FILL_DATE BETWEEN '2020-08-01' AND '2025-07-31'
  -- WHERE FILL_DATE BETWEEN '2020-04-01' AND '2025-03-31'
);

SELECT COUNT(DISTINCT patient_id) from MPSII_1Dx_Specified


MPS II 1 Dx - 1168

In [0]:
create or replace temporary view MPSII_2Dx_Specified
AS 
(SELECT DISTINCT PATIENT_ID AS PATIENT_ID 
FROM 
(
SELECT PATIENT_ID,  COUNT(DISTINCT FILL_DATE) AS NUMBER_OF_CLAIMS
FROM MPSII_1Dx_Specified
GROUP BY PATIENT_ID
)
WHERE NUMBER_OF_CLAIMS >=2);

SELECT COUNT(DISTINCT patient_id) FROM MPSII_2Dx_Specified

Elaprase Patients having MPS II Diagnosis and 2 Dx - 571

In [0]:
Select count(distinct patient_id) from MPSII_2Dx_Specified
where patient_id in (select distinct patient_id from MPSII_TREATMENT_TABLE);

In [0]:
create or replace temporary view MPSII_2Dx_Tx_Specified_Tx_claims AS
Select * from MPSII_TREATMENT_TABLE
where patient_id in (select distinct patient_id from MPSII_2Dx_Specified)
SELECT COUNT(DISTINCT patient_id) from MPSII_2Dx_Specified_Dx_claims

In [0]:
create or replace temporary view MPSII_2Dx_Specified_Dx_claims AS
Select * from MPSII_1Dx_Specified
where patient_id in (select distinct patient_id from MPSII_2Dx_Specified);
SELECT COUNT(DISTINCT patient_id) from MPSII_2Dx_Specified_Dx_claims